

---



# Transformers - Chess Assignment
### Mariyana Shishmanova

## Step 1 — Setting up and installing the libraries

In [1]:
!git clone https://github.com/bylinina/chess_exam.git
%cd /content/chess_exam
!pip install -e .
!pip install transformers accelerate bitsandbytes peft trl datasets chess requests zstandard

Cloning into 'chess_exam'...
remote: Enumerating objects: 268, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 268 (delta 69), reused 36 (delta 36), pack-reused 195 (from 2)
Receiving objects: 100% (268/268), 88.92 KiB | 1.29 MiB/s, done.
Resolving deltas: 100% (157/157), done.
/content/chess_exam
Obtaining file:///content/chess_exam
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 69.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of smolagents to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.1/533.1 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 12.0 MB/s eta 0:00:00
  Created wheel for chess: filename=chess-1.11.2

In [2]:
from chess_tournament import Game, Player, RandomPlayer, EnginePlayer
import chess
import chess.pgn
from typing import Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import login
import torch
import torch.nn.functional as F
import pandas as pd
import os
import io
import zstandard as zstd
import requests
from google.colab import files

Loging in to HuggingFace

In [ ]:
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Stockfish RapidAPI key

In [ ]:
from getpass import getpass
os.environ["RAPIDAPI_KEY"] = getpass("Paste your RapidAPI key:")

Paste your RapidAPI key:··········


## Step 2 — Generating training data

I decided to generate some traing data using the Stockfish model. I first did 100 games but that took ferever run and crashed, so i decided to stick to 20 games this time.

In [ ]:
engine_strong = EnginePlayer("Stockfish-Strong", blunder_rate=0.0, ponder_rate=0.05)

os.chdir('/content')
for i in range(20):
    game = Game(engine_strong, engine_strong, max_half_moves=80)
    game.play(log_moves=True, log_to_file=f'/content/stockfish_game_{i}.csv')
    print(f"Game {i+1}/20 done")

all_files = [f for f in os.listdir('/content') if f.startswith('stockfish_game_')]
dfs = [pd.read_csv(f'/content/{f}') for f in all_files]
combined = pd.concat(dfs, ignore_index=True)
combined.to_csv('/content/all_chess_data.csv', index=False)
print(f"✅ Stockfish data saved! Total moves: {len(combined)}")
files.download('/content/all_chess_data.csv')

PLY 000 | Stockfish-Strong | white | rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 | e2e4 | fallback:False
PLY 001 | Stockfish-Strong | white | rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1 | c7c5 | fallback:False
PLY 002 | Stockfish-Strong | white | rnbqkbnr/pp1ppppp/8/2p5/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2 | g1f3 | fallback:False
PLY 003 | Stockfish-Strong | white | rnbqkbnr/pp1ppppp/8/2p5/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2 | d7d6 | fallback:False
PLY 004 | Stockfish-Strong | white | rnbqkbnr/pp2pppp/3p4/2p5/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 0 3 | d2d4 | fallback:False
PLY 005 | Stockfish-Strong | white | rnbqkbnr/pp2pppp/3p4/2p5/3PP3/5N2/PPP2PPP/RNBQKB1R b KQkq - 0 3 | c5d4 | fallback:False
PLY 006 | Stockfish-Strong | white | rnbqkbnr/pp2pppp/3p4/8/3pP3/5N2/PPP2PPP/RNBQKB1R w KQkq - 0 4 | f3d4 | fallback:False
PLY 007 | Stockfish-Strong | white | rnbqkbnr/pp2pppp/3p4/8/3NP3/8/PPP2PPP/RNBQKB1R b KQkq - 0 4 | g8f6 | fallback:False
PLY 008 | Stockfish-St

KeyboardInterrupt: 

Since I only have 20 Stockfish games, I decided to also use some extra traing data. I downloaded the Lichess data which has over 5k moves.

In [ ]:
url = "https://database.lichess.org/standard/lichess_db_standard_rated_2013-01.pgn.zst"
print("Downloading lichess games...")
response = requests.get(url, stream=True)
with open('/content/lichess.pgn.zst', 'wb') as f:
    downloaded = 0
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        downloaded += len(chunk)
        if downloaded % (1024*1024) == 0:
            print(f"Downloaded: {downloaded // (1024*1024)} MB")
print("✅ Download done!")

Downloaded: 1 MB
Downloaded: 2 MB
Downloaded: 3 MB
Downloaded: 4 MB
Downloaded: 5 MB
Downloaded: 6 MB
Downloaded: 7 MB
Downloaded: 8 MB
Downloaded: 9 MB
Downloaded: 10 MB
Downloaded: 11 MB
Downloaded: 12 MB
Downloaded: 13 MB
Downloaded: 14 MB
Downloaded: 15 MB
Downloaded: 16 MB
✅ Download done!


Since I want the model to learn only good moves, I remove the ones that are not winning.

In [ ]:
rows = []
with open('/content/lichess.pgn.zst', 'rb') as f:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(f) as reader:
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')
        while len(rows) < 5000:
            game = chess.pgn.read_game(text_stream)
            if game is None:
                break
            result = game.headers.get("Result", "*")
            board = game.board()
            for move in game.mainline_moves():
                if result == "1-0" and board.turn == chess.WHITE:
                    rows.append({'fen_before': board.fen(), 'move': move.uci()})
                elif result == "0-1" and board.turn == chess.BLACK:
                    rows.append({'fen_before': board.fen(), 'move': move.uci()})
                board.push(move)

df_lichess = pd.DataFrame(rows)
print(f" Extracted {len(df_lichess)} winning moves from Lichess!")

✅ Extracted 5024 winning moves from Lichess!


I then decided to combine the Stockfish data that I generated and the Lichess data. Since my model will be tested specifically againdt Stockfish I though it would be nice to have both datasets. Hopefully my model is prepared to win against Stockfish.

In [ ]:
# Load Stockfish data and keep only good moves
df_stockfish = pd.read_csv('/content/all_chess_data.csv')
good_moves = df_stockfish[df_stockfish['fallback'] == False][['fen_before', 'move']]
print(f"Stockfish good moves: {len(good_moves)}")

# Combine both datasets
combined_df = pd.concat([good_moves, df_lichess], ignore_index=True)
print(f"Lichess winning moves: {len(df_lichess)}")
print(f"Total combined: {len(combined_df)}")

# Format for training
def format_row(row):
    return {"text": f"[INST] Chess move for: {row['fen_before']} [/INST] {row['move']}"}

formatted = combined_df.apply(format_row, axis=1).tolist()
formatted_df = pd.DataFrame(formatted)
dataset_combined = Dataset.from_pandas(formatted_df)

print(f"\n Dataset ready: {len(dataset_combined)} examples")
print("Example:", dataset_combined[0]['text'])

Stockfish good moves: 1169
Lichess winning moves: 5024
Total combined: 6193

✅ Dataset ready: 6193 examples
Example: [INST] Chess move for: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 [/INST] e2e4


## Step 3 — Loading TinyLlama and applying LoRA

In [ ]:
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Model ready for training!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
✅ Model ready for training!


## Step 4 — Train the model

In [ ]:
training_args = SFTConfig(
    output_dir="/content/chess_model_combined",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=20,
    save_steps=100,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_combined,
    args=training_args,
)

print("✅ Starting training...")
trainer.train()
print("✅ Training done!")

Adding EOS to train dataset:   0%|          | 0/6193 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6193 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6193 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


✅ Starting training...


Step,Training Loss
20,1.921179
40,1.132367
60,1.024487
80,0.960437
100,0.933883
120,0.954753
140,0.950633
160,0.937914
180,0.921290
200,0.882937


✅ Training done!


## Step 5 — Merge and push to HuggingFace


In [ ]:
merged_model = trainer.model.merge_and_unload()
print(" Model merged!")

hf_name = "Mariyana0019/chess-tinyllama-combined"
merged_model.push_to_hub(hf_name)
tokenizer.push_to_hub(hf_name)
print(f" Model saved to HuggingFace: {hf_name}")

NameError: name 'trainer' is not defined

## Step 6 — Building final TransformerPlayer


In [ ]:
class TransformerPlayer(Player):

    def __init__(self, name="MyPlayer"):
        super().__init__(name)

        model_name = "Mariyana0019/chess-tinyllama-combined"
        bnb_config = BitsAndBytesConfig(load_in_4bit=True)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print(f"✅ {self.name} is ready!")

    def score_move(self, prompt, move):
        full_text = prompt + " " + move
        inputs = self.tokenizer(full_text, return_tensors="pt").to("cuda")
        prompt_inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        prompt_length = prompt_inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            logits = outputs.logits[0, prompt_length-1:-1]
            labels = inputs["input_ids"][0, prompt_length:]
            loss = F.cross_entropy(logits, labels)
        return -loss.item()

    def get_move_bonus(self, board, move):
        bonus = 0.0
        if board.is_capture(move):
            bonus += 1.0
        board.push(move)
        if board.is_check():
            bonus += 0.5
        board.pop()
        center = ['e4', 'e5', 'd4', 'd5']
        if chess.square_name(move.to_square) in center:
            bonus += 0.3
        return bonus

    def get_move(self, fen):
        board = chess.Board(fen)
        legal_moves = list(board.legal_moves)
        if not legal_moves:
            return None
        prompt = f"[INST] Chess move for: {fen} [/INST]"
        best_move = None
        best_score = float('-inf')
        for move in legal_moves:
            model_score = self.score_move(prompt, move.uci())
            bonus = self.get_move_bonus(board, move)
            total_score = model_score + bonus
            if total_score > best_score:
                best_score = total_score
                best_move = move.uci()
        return best_move

my_player = TransformerPlayer("MyPlayer")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/410 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/807M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

✅ MyPlayer is ready!


## Step 7 — Testing the player

In [ ]:
wins, draws, losses, total_fallbacks = 0, 0, 0, 0

for i in range(5):
    r = Game(my_player, RandomPlayer("Random"), max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    fb = r[2].get("MyPlayer", 0)
    total_fallbacks += fb
    if points == 1.0:
        wins += 1
    elif points == 0.5:
        draws += 1
    else:
        losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}  |  fallbacks: {fb}")

print(f"\nSummary: {wins}W / {draws}D / {losses}L  |  Total fallbacks: {total_fallbacks}")

Game 1: DRAW  |  fallbacks: 0
Game 2: DRAW  |  fallbacks: 0
Game 3: DRAW  |  fallbacks: 0
Game 4: WIN  |  fallbacks: 0
Game 5: DRAW  |  fallbacks: 0

Summary: 1W / 4D / 0L  |  Total fallbacks: 0


Against the Random player it manages to win ones with no losses

In [ ]:
wins, draws, losses, total_fallbacks = 0, 0, 0, 0

for i in range(10):
    r = Game(my_player, RandomPlayer("Random"), max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    fb = r[2].get("MyPlayer", 0)
    total_fallbacks += fb
    if points == 1.0:
        wins += 1
    elif points == 0.5:
        draws += 1
    else:
        losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}  |  fallbacks: {fb}")

print(f"\nSummary: {wins}W / {draws}D / {losses}L  |  Total fallbacks: {total_fallbacks}")

Game 1: DRAW  |  fallbacks: 0
Game 2: DRAW  |  fallbacks: 0
Game 3: DRAW  |  fallbacks: 0
Game 4: DRAW  |  fallbacks: 0
Game 5: DRAW  |  fallbacks: 0
Game 6: DRAW  |  fallbacks: 0
Game 7: DRAW  |  fallbacks: 0
Game 8: DRAW  |  fallbacks: 0
Game 9: DRAW  |  fallbacks: 0
Game 10: DRAW  |  fallbacks: 0

Summary: 0W / 10D / 0L  |  Total fallbacks: 0


I ran it again for 10 games, this time no wins, only draw

**new strategy**

In [ ]:
class TransformerPlayer(Player):

    def __init__(self, name="MyPlayer"):
        super().__init__(name)

        model_name = "Mariyana0019/chess-tinyllama-combined"
        bnb_config = BitsAndBytesConfig(load_in_4bit=True)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print(f" {self.name} is ready!")

    def score_move(self, prompt, move):
        full_text = prompt + " " + move
        inputs = self.tokenizer(full_text, return_tensors="pt").to("cuda")
        prompt_inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        prompt_length = prompt_inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            logits = outputs.logits[0, prompt_length-1:-1]
            labels = inputs["input_ids"][0, prompt_length:]
            loss = F.cross_entropy(logits, labels)
        return -loss.item()

    def piece_value(self, board, move):
        # How valuable is the piece we're capturing?
        piece = board.piece_at(move.to_square)
        if piece is None:
            return 0
        values = {
            chess.PAWN: 1,
            chess.KNIGHT: 3,
            chess.BISHOP: 3,
            chess.ROOK: 5,
            chess.QUEEN: 9,
            chess.KING: 100
        }
        return values.get(piece.piece_type, 0)

    def get_move(self, fen):
        board = chess.Board(fen)
        legal_moves = list(board.legal_moves)

        if not legal_moves:
            return None

        # Priority 1: Checkmate — instant win, take it immediately!
        for move in legal_moves:
            board.push(move)
            if board.is_checkmate():
                board.pop()
                return move.uci()
            board.pop()

        # Priority 2: Best capture (take the most valuable piece)
        captures = [(move, self.piece_value(board, move))
                    for move in legal_moves if board.is_capture(move)]
        if captures:
            best_capture = max(captures, key=lambda x: x[1])
            if best_capture[1] >= 3:  # only take if it's worth at least a knight
                return best_capture[0].uci()

        # Priority 3: Give check
        for move in legal_moves:
            board.push(move)
            if board.is_check():
                board.pop()
                return move.uci()
            board.pop()

        # Priority 4: Use model score for everything else
        prompt = f"[INST] Chess move for: {fen} [/INST]"
        best_move = None
        best_score = float('-inf')
        for move in legal_moves:
            score = self.score_move(prompt, move.uci())
            if score > best_score:
                best_score = score
                best_move = move.uci()

        return best_move

my_player = TransformerPlayer("MyPlayer")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ MyPlayer is ready!


In [ ]:
wins, draws, losses, total_fallbacks = 0, 0, 0, 0

for i in range(5):
    r = Game(my_player, RandomPlayer("Random"), max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    fb = r[2].get("MyPlayer", 0)
    total_fallbacks += fb

    if points == 1.0:
        wins += 1
    elif points == 0.5:
        draws += 1
    else:
        losses += 1

    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}  |  fallbacks: {fb}")

print(f"\nSummary: {wins}W / {draws}D / {losses}L  |  Total fallbacks: {total_fallbacks}")

Game 1: DRAW  |  fallbacks: 0
Game 2: DRAW  |  fallbacks: 0
Game 3: DRAW  |  fallbacks: 0
Game 4: WIN  |  fallbacks: 0
Game 5: WIN  |  fallbacks: 0

Summary: 2W / 3D / 0L  |  Total fallbacks: 0


# **Test**

In [3]:
%cd /content
from chess_tournament import validate_player

res = validate_player('https://github.com/MS19MS/Transformers-Chess.git')
print(res)

/content
Cloning https://github.com/MS19MS/Transformers-Chess.git...
✓ Clone successful
Installing requirements.txt...
✓ Requirements installed
Validating Transformers-Chess...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/807M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

✅ student_test is ready!
Cleaning up Transformers-Chess...
{'import_ok': True, 'class_found': True, 'instance_ok': True, 'move_from_fen': None, 'valid_move_format': False, 'error_message': 'get_move_exception:\nTraceback (most recent call last):\n  File "/content/chess_exam/chess_tournament/validate.py", line 74, in _validate_local\n    move = inst.get_move(fen)\n           ^^^^^^^^^^^^^^^^^^\n  File "/content/Transformers-Chess/player.py", line 148, in get_move\n    model_score = self.score_move(prompt, move.uci())\n                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/content/Transformers-Chess/player.py", line 35, in score_move\n    inputs = self.tokenizer(full_text, return_tensors="pt").to("cuda")\n             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 774, in to\n    k: v.to(device=device, non_blocking=non_blocking) if hasattr(v, "to") and callable(v.to) else v

In [4]:
%cd /content/Transformers-Chess
!ls -la

[Errno 2] No such file or directory: '/content/Transformers-Chess'
/content
total 20
drwxr-xr-x 1 root root 4096 Mar  8 19:49 .
drwxr-xr-x 1 root root 4096 Mar  8 19:30 ..
drwxr-xr-x 6 root root 4096 Mar  8 19:35 chess_exam
drwxr-xr-x 4 root root 4096 Jan 16 14:24 .config
drwxr-xr-x 1 root root 4096 Jan 16 14:24 sample_data


In [5]:
!rm -rf /content/Transformers-Chess
!git clone https://github.com/MS19MS/Transformers-Chess.git

import sys
sys.path.insert(0, '/content/Transformers-Chess')

from chess_tournament import Game, RandomPlayer, EnginePlayer
from chess_player import TransformerPlayer

my_player = TransformerPlayer("MyPlayer")

Cloning into 'Transformers-Chess'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 24 (delta 8), reused 19 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 2.39 MiB | 3.21 MiB/s, done.
Resolving deltas: 100% (8/8), done.


ModuleNotFoundError: No module named 'chess_player'

In [6]:
from chess_tournament import Game, RandomPlayer, EnginePlayer

my_player = TransformerPlayer("MyPlayer")
# Test 3 — vs Strong Stockfish (the real challenge)
engine_strong = EnginePlayer("Stockfish-Strong", blunder_rate=0.0, ponder_rate=0.05)

wins, draws, losses = 0, 0, 0
for i in range(5):
    r = Game(my_player, engine_strong, max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    if points == 1.0: wins += 1
    elif points == 0.5: draws += 1
    else: losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}")

print(f"\nVS Strong Stockfish → {wins}W / {draws}D / {losses}L")

NameError: name 'TransformerPlayer' is not defined

# ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


# Traning a second model with more data

MOre data

In [ ]:
# Cell 1
!git clone https://github.com/bylinina/chess_exam.git

Cloning into 'chess_exam'...
remote: Enumerating objects: 268, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 268 (delta 69), reused 36 (delta 36), pack-reused 195 (from 2)
Receiving objects: 100% (268/268), 88.92 KiB | 752.00 KiB/s, done.
Resolving deltas: 100% (157/157), done.


In [ ]:
%cd /content/chess_exam
!pip install -e .

/content/chess_exam
Obtaining file:///content/chess_exam
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 83.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of smolagents to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.1/533.1 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 17.8 MB/s eta 0:00:00
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=aa3982bdd1eebb2574304f2b99fbcadcb048e73def80145770cd4c453729f6af
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling 

In [ ]:

!pip install transformers accelerate bitsandbytes peft trl datasets chess requests zstandard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 4.4 MB/s eta 0:00:00


In [ ]:

from chess_tournament import Game, Player, RandomPlayer, EnginePlayer
print("✅ chess_tournament imported!")

✅ chess_tournament imported!


In [ ]:




import chess
import chess.pgn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import login
import torch
import torch.nn.functional as F
import pandas as pd
import os
import io
import zstandard as zstd
import requests
print("✅ Imports done!")





✅ Imports done!


In [ ]:
# Download Lichess data
url = "https://database.lichess.org/standard/lichess_db_standard_rated_2013-01.pgn.zst"
print("Downloading lichess games...")
response = requests.get(url, stream=True)
with open('/content/lichess.pgn.zst', 'wb') as f:
    downloaded = 0
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        downloaded += len(chunk)
        if downloaded % (5*1024*1024) == 0:
            print(f"Downloaded: {downloaded // (1024*1024)} MB")
print(" Download done!")

In [ ]:
# Extract 50,000 moves
rows = []
games_processed = 0

with open('/content/lichess.pgn.zst', 'rb') as f:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(f) as reader:
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')
        while len(rows) < 50000:
            game = chess.pgn.read_game(text_stream)
            if game is None:
                break

            # Only use games from strong players
            try:
                white_elo = int(game.headers.get("WhiteElo", 0))
                black_elo = int(game.headers.get("BlackElo", 0))
            except:
                continue

            if white_elo < 2000 or black_elo < 2000:
                continue  # skip weak players

            result = game.headers.get("Result", "*")
            if result not in ["1-0", "0-1"]:
                continue  # skip draws

            board = game.board()
            for move in game.mainline_moves():
                if result == "1-0" and board.turn == chess.WHITE:
                    rows.append({'fen_before': board.fen(), 'move': move.uci()})
                elif result == "0-1" and board.turn == chess.BLACK:
                    rows.append({'fen_before': board.fen(), 'move': move.uci()})
                board.push(move)

            games_processed += 1
            if games_processed % 100 == 0:
                print(f"Games processed: {games_processed} | Moves collected: {len(rows)}")

df_lichess = pd.DataFrame(rows)
print(f"✅ Extracted {len(df_lichess)} grandmaster moves!")

Games processed: 100 | Moves collected: 3859
Games processed: 200 | Moves collected: 7738
Games processed: 300 | Moves collected: 11766
Games processed: 400 | Moves collected: 15627
Games processed: 500 | Moves collected: 19335
✅ Extracted 21894 grandmaster moves!


In [ ]:
# Download a bigger database

url2 = "https://database.lichess.org/standard/lichess_db_standard_rated_2014-06.pgn.zst"
print("Downloading second database...")
response = requests.get(url2, stream=True)
with open('/content/lichess2.pgn.zst', 'wb') as f:
    downloaded = 0
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        downloaded += len(chunk)
        if downloaded % (5*1024*1024) == 0:
            print(f"Downloaded: {downloaded // (1024*1024)} MB")
print(" Second database downloaded!")

Downloaded: 5 MB
Downloaded: 10 MB
Downloaded: 15 MB
Downloaded: 20 MB
Downloaded: 25 MB
Downloaded: 30 MB
Downloaded: 35 MB
Downloaded: 40 MB
Downloaded: 45 MB
Downloaded: 50 MB
Downloaded: 55 MB
Downloaded: 60 MB
Downloaded: 65 MB
Downloaded: 70 MB
Downloaded: 75 MB
Downloaded: 80 MB
Downloaded: 85 MB
Downloaded: 90 MB
Downloaded: 95 MB
Downloaded: 100 MB
Downloaded: 105 MB
Downloaded: 110 MB
Downloaded: 115 MB
Downloaded: 120 MB
Downloaded: 125 MB
Downloaded: 130 MB
Downloaded: 135 MB
Downloaded: 140 MB
Downloaded: 145 MB
Downloaded: 150 MB
Downloaded: 155 MB
Downloaded: 160 MB
Downloaded: 165 MB
Downloaded: 170 MB
✅ Second database downloaded!


In [ ]:
# Extract from second database
rows2 = []
games_processed2 = 0

with open('/content/lichess2.pgn.zst', 'rb') as f:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(f) as reader:
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')
        while len(rows2) < 30000:  # get 30k more to top up to 50k total
            game = chess.pgn.read_game(text_stream)
            if game is None:
                break
            try:
                white_elo = int(game.headers.get("WhiteElo", 0))
                black_elo = int(game.headers.get("BlackElo", 0))
            except:
                continue
            if white_elo < 1800 or black_elo < 1800:
                continue
            result = game.headers.get("Result", "*")
            if result not in ["1-0", "0-1"]:
                continue
            board = game.board()
            for move in game.mainline_moves():
                if result == "1-0" and board.turn == chess.WHITE:
                    rows2.append({'fen_before': board.fen(), 'move': move.uci()})
                elif result == "0-1" and board.turn == chess.BLACK:
                    rows2.append({'fen_before': board.fen(), 'move': move.uci()})
                board.push(move)
            games_processed2 += 1
            if games_processed2 % 100 == 0:
                print(f"Games: {games_processed2} | Moves: {len(rows2)}")

df_lichess2 = pd.DataFrame(rows2)
print(f" Second database: {len(df_lichess2)} moves!")

Games: 100 | Moves: 3622
Games: 200 | Moves: 7213
Games: 300 | Moves: 10902
Games: 400 | Moves: 14766
Games: 500 | Moves: 18389
Games: 600 | Moves: 21893
Games: 700 | Moves: 25192
Games: 800 | Moves: 29024
✅ Second database: 30020 moves!


In [ ]:
# Combine both databases
df_lichess_combined = pd.concat([df_lichess, df_lichess2], ignore_index=True)
df_lichess_combined = df_lichess_combined.drop_duplicates()  # remove any duplicate positions
print(f"✅ Total combined: {len(df_lichess_combined)} moves!")

# Replace df_lichess with the combined version for the next steps
df_lichess = df_lichess_combined

✅ Total combined: 48539 moves!


In [ ]:
#  Combine Stockfish + Lichess data
df_stockfish = pd.read_csv('/content/all_chess_data.csv')
good_moves = df_stockfish[df_stockfish['fallback'] == False][['fen_before', 'move']]
print(f"Stockfish good moves: {len(good_moves)}")

combined_df = pd.concat([good_moves, df_lichess], ignore_index=True)
combined_df = combined_df.drop_duplicates()
print(f"Total combined: {len(combined_df)}")

# Format for training
def format_row(row):
    return {"text": f"[INST] Chess move for: {row['fen_before']} [/INST] {row['move']}"}

formatted = combined_df.apply(format_row, axis=1).tolist()
formatted_df = pd.DataFrame(formatted)
dataset_combined = Dataset.from_pandas(formatted_df)

print(f" Dataset ready: {len(dataset_combined)} examples")
print("Example:", dataset_combined[0]['text'])

Stockfish good moves: 1169
Total combined: 49412
✅ Dataset ready: 49412 examples
Example: [INST] Chess move for: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 [/INST] e2e4


In [ ]:

combined_df.to_csv('/content/combined_dataset.csv', index=False)
from google.colab import files
files.download('/content/combined_dataset.csv')
print(" Dataset saved!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Dataset saved!


In [ ]:
# Load TinyLlama and apply LoRA
from huggingface_hub import login
login()

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Model ready for training!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
✅ Model ready for training!


In [ ]:
# Load saved dataset
import pandas as pd
from datasets import Dataset

combined_df = pd.read_csv('/content/combined_dataset.csv')
print(f"✅ Loaded {len(combined_df)} moves!")

# Format for training
def format_row(row):
    return {"text": f"[INST] Chess move for: {row['fen_before']} [/INST] {row['move']}"}

formatted = combined_df.apply(format_row, axis=1).tolist()
dataset_combined = Dataset.from_pandas(pd.DataFrame(formatted))
print(f"✅ Dataset ready: {len(dataset_combined)} examples")
print("Example:", dataset_combined[0]['text'])

✅ Loaded 49412 moves!
✅ Dataset ready: 49412 examples
Example: [INST] Chess move for: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 [/INST] e2e4


In [ ]:
training_args = SFTConfig(
    output_dir="/content/chess_model_ultimate",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_steps=500,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
)

dataset_small = dataset_combined.select(range(20000))

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_small,
    args=training_args,
)

trainer.train()
print("✅ Done!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Step,Training Loss
50,1.740344
100,1.059921
150,1.008925
200,0.978951
250,0.969718
300,0.952017
350,0.926288
400,0.920410
450,0.912813
500,0.914698


✅ Done!


In [ ]:
# Step 11 — Push to HuggingFace (run tomorrow when training finishes)
merged_model = trainer.model.merge_and_unload()
hf_name = "Mariyana0019/chess-tinyllama-ultimate"
merged_model.push_to_hub(hf_name)
tokenizer.push_to_hub(hf_name)
print(f"✅ Model saved to: https://huggingface.co/{hf_name}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...cl6g8kf/model.safetensors:   4%|3         | 29.7MB /  807MB            

README.md: 0.00B [00:00, ?B/s]

✅ Model saved to: https://huggingface.co/Mariyana0019/chess-tinyllama-ultimate


The model has been trained and is on HF, now let's test it

In [ ]:
# Load your new ultimate model
import sys
sys.path.insert(0, '/content/Transformers-Chess')

from chess_tournament import Game, RandomPlayer, EnginePlayer, Player
import chess
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import torch.nn.functional as F

class TransformerPlayer(Player):

    def __init__(self, name="MyPlayer"):
        super().__init__(name)
        model_name = "Mariyana0019/chess-tinyllama-ultimate"
        bnb_config = BitsAndBytesConfig(load_in_4bit=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print(f" {self.name} is ready!")

    def score_move(self, prompt, move):
        full_text = prompt + " " + move
        inputs = self.tokenizer(full_text, return_tensors="pt").to("cuda")
        prompt_inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        prompt_length = prompt_inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            logits = outputs.logits[0, prompt_length-1:-1]
            labels = inputs["input_ids"][0, prompt_length:]
            loss = F.cross_entropy(logits, labels)
        return -loss.item()

    def piece_value(self, piece_type):
        return {chess.PAWN:1, chess.KNIGHT:3, chess.BISHOP:3,
                chess.ROOK:5, chess.QUEEN:9, chess.KING:100}.get(piece_type, 0)

    def is_hanging(self, board, square):
        piece = board.piece_at(square)
        if piece is None:
            return False
        attackers = board.attackers(not piece.color, square)
        defenders = board.attackers(piece.color, square)
        return len(attackers) > len(defenders)

    def would_hang_piece(self, board, move):
        board.push(move)
        our_color = not board.turn
        hanging = False
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece and piece.color == our_color:
                if self.is_hanging(board, square):
                    hanging = True
                    break
        board.pop()
        return hanging

    def get_move_bonus(self, board, move):
        bonus = 0.0
        if board.is_capture(move):
            captured = board.piece_at(move.to_square)
            if captured:
                bonus += self.piece_value(captured.piece_type) * 0.5
        board.push(move)
        if board.is_check():
            bonus += 1.0
        board.pop()
        center = [chess.E4, chess.E5, chess.D4, chess.D5]
        extended = [chess.C3, chess.C4, chess.C5, chess.C6,
                    chess.D3, chess.D6, chess.E3, chess.E6,
                    chess.F3, chess.F4, chess.F5, chess.F6]
        if move.to_square in center:
            bonus += 0.5
        elif move.to_square in extended:
            bonus += 0.2
        piece = board.piece_at(move.from_square)
        if piece and piece.piece_type == chess.KING:
            if board.is_castling(move):
                bonus += 2.0
        board.push(move)
        moved_piece = board.piece_at(move.to_square)
        if moved_piece:
            attackers = board.attackers(not moved_piece.color, move.to_square)
            if attackers:
                bonus -= self.piece_value(moved_piece.piece_type) * 0.3
        board.pop()
        if self.would_hang_piece(board, move):
            bonus -= 2.0
        return bonus

    def get_move(self, fen):
        board = chess.Board(fen)
        legal_moves = list(board.legal_moves)
        if not legal_moves:
            return None
        for move in legal_moves:
            board.push(move)
            if board.is_checkmate():
                board.pop()
                return move.uci()
            board.pop()
        for move in legal_moves:
            if board.is_capture(move):
                captured = board.piece_at(move.to_square)
                if captured and not self.would_hang_piece(board, move):
                    if self.piece_value(captured.piece_type) >= 3:
                        return move.uci()
        prompt = f"[INST] Chess move for: {fen} [/INST]"
        best_move = None
        best_score = float('-inf')
        for move in legal_moves:
            model_score = self.score_move(prompt, move.uci())
            bonus = self.get_move_bonus(board, move)
            total = model_score + bonus
            if total > best_score:
                best_score = total
                best_move = move.uci()
        return best_move

my_player = TransformerPlayer("MyPlayer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/410 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/807M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

✅ MyPlayer is ready!


lets test it against Stockfish first first with the weak one

In [ ]:
# Test vs Weak Stockfish
wins, draws, losses = 0, 0, 0
engine_weak = EnginePlayer("Stockfish-Weak", blunder_rate=0.5, ponder_rate=0.0)
for i in range(5):
    r = Game(my_player, engine_weak, max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    if points == 1.0: wins += 1
    elif points == 0.5: draws += 1
    else: losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}")
print(f"\nVS Weak Stockfish → {wins}W / {draws}D / {losses}L")

Game 1: DRAW
Game 2: WIN
Game 3: WIN
Game 4: DRAW
Game 5: DRAW

VS Weak Stockfish → 2W / 3D / 0L


Testing with Strong Stockfish

In [ ]:
# Test vs Strong Stockfish
wins, draws, losses = 0, 0, 0
engine_strong = EnginePlayer("Stockfish-Strong", blunder_rate=0.0, ponder_rate=0.05)
for i in range(5):
    r = Game(my_player, engine_strong, max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    if points == 1.0: wins += 1
    elif points == 0.5: draws += 1
    else: losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}")
print(f"\nVS Strong Stockfish → {wins}W / {draws}D / {losses}L")

Game 1: WIN
Game 2: DRAW
Game 3: LOSS
Game 4: LOSS
Game 5: DRAW

VS Strong Stockfish → 1W / 2D / 2L


Testing with Random

In [ ]:
# Test vs Random (should win easily)
wins, draws, losses = 0, 0, 0
for i in range(5):
    r = Game(my_player, RandomPlayer("Random"), max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    if points == 1.0: wins += 1
    elif points == 0.5: draws += 1
    else: losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}")
print(f"\nVS Random → {wins}W / {draws}D / {losses}L")

Game 1: WIN
Game 2: WIN
Game 3: WIN
Game 4: WIN
Game 5: WIN

VS Random → 5W / 0D / 0L


Improving the instructions

In [14]:
from chess_tournament import Player
import chess
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import torch.nn.functional as F


class TransformerPlayer(Player):

    def __init__(self, name="MyPlayer"):
        super().__init__(name)
        model_name = "Mariyana0019/chess-tinyllama-ultimate"
        bnb_config = BitsAndBytesConfig(load_in_4bit=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
        self.move_history = []
        print(f"✅ {self.name} is ready!")

    def score_move(self, prompt, move):
        full_text = prompt + " " + move
        inputs = self.tokenizer(full_text, return_tensors="pt").to("cuda")
        prompt_inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        prompt_length = prompt_inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            logits = outputs.logits[0, prompt_length-1:-1]
            labels = inputs["input_ids"][0, prompt_length:]
            loss = F.cross_entropy(logits, labels)
        return -loss.item()

    def piece_value(self, piece_type):
        return {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
                chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 100}.get(piece_type, 0)

    def is_hanging(self, board, square):
        piece = board.piece_at(square)
        if piece is None:
            return False
        attackers = board.attackers(not piece.color, square)
        defenders = board.attackers(piece.color, square)
        return len(attackers) > len(defenders)

    def would_hang_piece(self, board, move):
        board.push(move)
        our_color = not board.turn
        hanging = False
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece and piece.color == our_color:
                if self.is_hanging(board, square):
                    hanging = True
                    break
        board.pop()
        return hanging

    def is_good_trade(self, board, move):
        if not board.is_capture(move):
            return True
        captured = board.piece_at(move.to_square)
        attacker = board.piece_at(move.from_square)
        if captured and attacker:
            return self.piece_value(captured.piece_type) >= self.piece_value(attacker.piece_type)
        return True

    def is_safe_move(self, board, move):
        board.push(move)
        for opp_move in board.legal_moves:
            board.push(opp_move)
            if board.is_checkmate():
                board.pop()
                board.pop()
                return False
            board.pop()
        board.pop()
        return True

    def king_is_exposed(self, board, move):
        board.push(move)
        our_color = not board.turn
        king_square = board.king(our_color)
        if king_square is None:
            board.pop()
            return False
        attackers = len(board.attackers(board.turn, king_square))
        board.pop()
        return attackers > 0

    def get_move_bonus(self, board, move):
        bonus = 0.0

        # Reward captures weighted by piece value
        if board.is_capture(move):
            captured = board.piece_at(move.to_square)
            if captured:
                bonus += self.piece_value(captured.piece_type) * 0.5

        # Reward checks
        board.push(move)
        if board.is_check():
            bonus += 1.0
        board.pop()

        # Reward center control
        center = [chess.E4, chess.E5, chess.D4, chess.D5]
        extended = [chess.C3, chess.C4, chess.C5, chess.C6,
                    chess.D3, chess.D6, chess.E3, chess.E6,
                    chess.F3, chess.F4, chess.F5, chess.F6]
        if move.to_square in center:
            bonus += 0.5
        elif move.to_square in extended:
            bonus += 0.2

        # Reward castling
        piece = board.piece_at(move.from_square)
        if piece and piece.piece_type == chess.KING:
            if board.is_castling(move):
                bonus += 2.0

        # Penalize moving into attacks
        board.push(move)
        moved_piece = board.piece_at(move.to_square)
        if moved_piece:
            attackers = board.attackers(not moved_piece.color, move.to_square)
            if attackers:
                bonus -= self.piece_value(moved_piece.piece_type) * 0.3
        board.pop()

        # Penalize hanging pieces
        if self.would_hang_piece(board, move):
            bonus -= 2.0

        return bonus

    def get_move(self, fen):
        board = chess.Board(fen)
        legal_moves = list(board.legal_moves)
        if not legal_moves:
            return None

        # Priority 1: Checkmate immediately
        for move in legal_moves:
            board.push(move)
            if board.is_checkmate():
                board.pop()
                return move.uci()
            board.pop()

        # Priority 2: Promote to queen
        for move in legal_moves:
            if move.promotion == chess.QUEEN:
                return move.uci()

        # Priority 3: Free captures (good trades only, no hanging after)
        for move in legal_moves:
            if board.is_capture(move):
                captured = board.piece_at(move.to_square)
                if captured and self.piece_value(captured.piece_type) >= 3:
                    if self.is_good_trade(board, move):
                        if not self.would_hang_piece(board, move):
                            return move.uci()

        # Priority 4: Filter out moves that walk into checkmate
        safe_moves = [m for m in legal_moves if self.is_safe_move(board, m)]
        moves_to_score = safe_moves if safe_moves else legal_moves

        # Priority 5: Model score + bonuses
        prompt = f"[INST] Chess move for: {fen} [/INST]"
        best_move = None
        best_score = float('-inf')

        for move in moves_to_score:
            model_score = self.score_move(prompt, move.uci())
            bonus = self.get_move_bonus(board, move)

            # Penalize king exposure
            if self.king_is_exposed(board, move):
                bonus -= 1.5

            # Penalize repeated moves
            if move.uci() in self.move_history[-6:]:
                bonus -= 1.0

            total = model_score + bonus
            if total > best_score:
                best_score = total
                best_move = move.uci()

        if best_move:
            self.move_history.append(best_move)

        return best_move

lets try this one now with the strong stockfish

In [15]:
engine_strong = EnginePlayer("Stockfish-Strong", blunder_rate=0.0, ponder_rate=0.05)

wins, draws, losses = 0, 0, 0

for i in range(5):
    r = Game(my_player, engine_strong, max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    if points == 1.0: wins += 1
    elif points == 0.5: draws += 1
    else: losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}")

print(f"\nVS Strong Stockfish → {wins}W / {draws}D / {losses}L")

Game 1: LOSS
Game 2: LOSS
Game 3: LOSS
Game 4: LOSS
Game 5: LOSS

VS Strong Stockfish → 0W / 0D / 5L


Improving it made it worse, lets try with the weak stockfish too

In [16]:
# Test vs Weak Stockfish
wins, draws, losses = 0, 0, 0
engine_weak = EnginePlayer("Stockfish-Weak", blunder_rate=0.5, ponder_rate=0.0)
for i in range(5):
    r = Game(my_player, engine_weak, max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    if points == 1.0: wins += 1
    elif points == 0.5: draws += 1
    else: losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}")
print(f"\nVS Weak Stockfish → {wins}W / {draws}D / {losses}L")

Game 1: WIN
Game 2: DRAW
Game 3: DRAW
Game 4: WIN
Game 5: WIN

VS Weak Stockfish → 3W / 2D / 0L


I removed is_safe_move and king_is_exposed because they were too aggressive in filtering moves and causing it to make worse choices

In [17]:

class TransformerPlayer(Player):

    def __init__(self, name="MyPlayer"):
        super().__init__(name)
        model_name = "Mariyana0019/chess-tinyllama-ultimate"
        bnb_config = BitsAndBytesConfig(load_in_4bit=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
        self.move_history = []
        print(f"✅ {self.name} is ready!")

    def score_move(self, prompt, move):
        full_text = prompt + " " + move
        inputs = self.tokenizer(full_text, return_tensors="pt").to("cuda")
        prompt_inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        prompt_length = prompt_inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            logits = outputs.logits[0, prompt_length-1:-1]
            labels = inputs["input_ids"][0, prompt_length:]
            loss = F.cross_entropy(logits, labels)
        return -loss.item()

    def piece_value(self, piece_type):
        return {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
                chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 100}.get(piece_type, 0)

    def would_hang_piece(self, board, move):
        board.push(move)
        our_color = not board.turn
        hanging = False
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece and piece.color == our_color:
                attackers = board.attackers(not our_color, square)
                defenders = board.attackers(our_color, square)
                if len(attackers) > len(defenders):
                    hanging = True
                    break
        board.pop()
        return hanging

    def is_good_trade(self, board, move):
        if not board.is_capture(move):
            return True
        captured = board.piece_at(move.to_square)
        attacker = board.piece_at(move.from_square)
        if captured and attacker:
            return self.piece_value(captured.piece_type) >= self.piece_value(attacker.piece_type)
        return True

    def get_move_bonus(self, board, move):
        bonus = 0.0

        # Reward captures weighted by piece value
        if board.is_capture(move):
            captured = board.piece_at(move.to_square)
            if captured:
                bonus += self.piece_value(captured.piece_type) * 0.5

        # Reward checks
        board.push(move)
        if board.is_check():
            bonus += 1.0
        board.pop()

        # Reward center control
        center = [chess.E4, chess.E5, chess.D4, chess.D5]
        extended = [chess.C3, chess.C4, chess.C5, chess.C6,
                    chess.D3, chess.D6, chess.E3, chess.E6,
                    chess.F3, chess.F4, chess.F5, chess.F6]
        if move.to_square in center:
            bonus += 0.5
        elif move.to_square in extended:
            bonus += 0.2

        # Reward castling
        piece = board.piece_at(move.from_square)
        if piece and piece.piece_type == chess.KING:
            if board.is_castling(move):
                bonus += 2.0

        # Penalize hanging pieces
        if self.would_hang_piece(board, move):
            bonus -= 2.0

        # Penalize repeated moves
        if move.uci() in self.move_history[-6:]:
            bonus -= 1.0

        return bonus

    def get_move(self, fen):
        board = chess.Board(fen)
        legal_moves = list(board.legal_moves)
        if not legal_moves:
            return None

        # Priority 1: Checkmate immediately
        for move in legal_moves:
            board.push(move)
            if board.is_checkmate():
                board.pop()
                return move.uci()
            board.pop()

        # Priority 2: Promote to queen
        for move in legal_moves:
            if move.promotion == chess.QUEEN:
                return move.uci()

        # Priority 3: Good captures only
        for move in legal_moves:
            if board.is_capture(move):
                captured = board.piece_at(move.to_square)
                if captured and self.piece_value(captured.piece_type) >= 3:
                    if self.is_good_trade(board, move):
                        if not self.would_hang_piece(board, move):
                            return move.uci()

        # Priority 4: Model score + bonuses
        prompt = f"[INST] Chess move for: {fen} [/INST]"
        best_move = None
        best_score = float('-inf')

        for move in legal_moves:
            model_score = self.score_move(prompt, move.uci())
            bonus = self.get_move_bonus(board, move)
            total = model_score + bonus
            if total > best_score:
                best_score = total
                best_move = move.uci()

        if best_move:
            self.move_history.append(best_move)

        return best_move

In [18]:
engine_strong = EnginePlayer("Stockfish-Strong", blunder_rate=0.0, ponder_rate=0.05)

wins, draws, losses = 0, 0, 0

for i in range(5):
    r = Game(my_player, engine_strong, max_half_moves=300).play()
    points = r[1]["MyPlayer"]
    if points == 1.0: wins += 1
    elif points == 0.5: draws += 1
    else: losses += 1
    print(f"Game {i+1}: {'WIN' if points==1.0 else 'DRAW' if points==0.5 else 'LOSS'}")

print(f"\nVS Strong Stockfish → {wins}W / {draws}D / {losses}L")

Game 1: DRAW
Game 2: DRAW
Game 3: DRAW
Game 4: LOSS
Game 5: LOSS

VS Strong Stockfish → 0W / 3D / 2L


# Test

In [9]:
from chess_tournament import validate_player

res = validate_player('https://github.com/MS19MS/Transformers-Chess.git')
print(res)

Cloning https://github.com/MS19MS/Transformers-Chess.git...
✓ Clone successful
Installing requirements.txt...
✓ Requirements installed
Validating Transformers-Chess...


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ student_test is ready!
Cleaning up Transformers-Chess...
{'import_ok': True, 'class_found': True, 'instance_ok': True, 'move_from_fen': None, 'valid_move_format': False, 'error_message': 'get_move_exception:\nTraceback (most recent call last):\n  File "/content/chess_exam/chess_tournament/validate.py", line 74, in _validate_local\n    move = inst.get_move(fen)\n           ^^^^^^^^^^^^^^^^^^\n  File "/content/Transformers-Chess/player.py", line 148, in get_move\n    model_score = self.score_move(prompt, move.uci())\n                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/content/Transformers-Chess/player.py", line 35, in score_move\n    inputs = self.tokenizer(full_text, return_tensors="pt").to("cuda")\n             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 774, in to\n    k: v.to(device=device, non_blocking=non_blocking) if hasattr(v, "to") and callable(v.to) else v